# TACTIC-FP 01 - GPU Trajectory NPZ Generation

This notebook is the **next phase after annotation export**. It turns a clean `_TRAIN.json` intent manifest plus a match video into per-segment trajectory tensors:

```text
raw_videos/match_001_720p.mp4
+ data/exports/TACTIC_FP_Annotated_match_001_TRAIN.json
-> data/trajectories/match_001/*.npz
```

Each non-excluded segment receives one `.npz` containing a `trajectory` array with shape `[T, 23, 4]`:

- agents `0..10`: team A players
- agents `11..21`: team B players
- agent `22`: ball
- channels: normalized `x, y, dx, dy`

Excluded segments such as `DeadBall` and `ContestedPlay` are skipped by model training and do not need NPZ files.

**Important:** This notebook does not generate fake/synthetic trajectories. If detection quality is poor, it writes warnings so the segment can be reviewed or regenerated.

## 0. Recommended Notebook Split

For conference submission, keep notebooks separate:

1. `TACTIC_FP_01_Trajectory_NPZ_Generation.ipynb` - deterministic dataset preparation and validation.
2. `TACTIC_FP_02_Model_Training.ipynb` - model training, ablations, metrics, and plots.

This separation makes the artifact easier to review and reproduce. Training code should consume already-validated `.json + .npz` files rather than mixing video processing with model optimization.

In [ ]:
# Optional installs. Run this cell only if imports below fail.
# In a fresh GPU environment, uncomment and run:
# %pip install -U ultralytics opencv-python numpy pandas tqdm scikit-learn

In [ ]:
from pathlib import Path
import json
import math
import os
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception as exc:
    torch = None
    print("Torch import failed:", exc)

try:
    from ultralytics import YOLO
except Exception as exc:
    YOLO = None
    print("Ultralytics import failed. Install with: %pip install -U ultralytics")
    print(exc)

try:
    from sklearn.cluster import KMeans
except Exception as exc:
    KMeans = None
    print("scikit-learn import failed. Team clustering will use x-position fallback.")
    print(exc)

In [ ]:
# ------------------------------
# User configuration
# ------------------------------
PROJECT_ROOT = Path.cwd()
TRAIN_JSON_PATH = PROJECT_ROOT / "data" / "exports" / "TACTIC_FP_Annotated_match_001_TRAIN.json"

# Prefer browser-compatible converted video when present.
VIDEO_PATH = PROJECT_ROOT / "raw_videos" / "match_001_720p.mp4"
if not VIDEO_PATH.exists():
    VIDEO_PATH = PROJECT_ROOT / "raw_videos" / "match_001.mkv"

# YOLO model. yolo11n is fast; yolo11s/yolo11m can improve quality if GPU memory allows.
YOLO_MODEL_NAME = "yolo11n.pt"
DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
IMG_SIZE = 1280
CONF_PERSON = 0.25
CONF_BALL = 0.08
IOU = 0.55
TARGET_FPS = 10
MAX_AGENTS = 23
MAX_PLAYERS = 22
MAX_FRAMES = 150

# If broadcast video includes pre-match footage, set this offset so match clock 0 maps to video time.
VIDEO_TIME_OFFSET_SEC = 0.0

# Team assignment mode:
# - "color": use jersey-color clustering from player crops, then x-position ordering as a fallback.
# - "x": split players by x coordinate only. Less reliable, but dependency-free.
TEAM_ASSIGNMENT_MODE = "color"

# Output behavior
OVERWRITE_EXISTING_NPZ = True
REPORT_PATH = PROJECT_ROOT / "data" / "exports" / "trajectory_generation_report.csv"

print("Project root:", PROJECT_ROOT)
print("Train JSON:", TRAIN_JSON_PATH, TRAIN_JSON_PATH.exists())
print("Video:", VIDEO_PATH, VIDEO_PATH.exists())
print("Device:", DEVICE)
if torch is not None and torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ------------------------------
# Utility and validation helpers
# ------------------------------
def load_train_manifest(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    required_root = {"match_id", "model_split", "halves"}
    missing = required_root - set(data)
    if missing:
        raise ValueError(f"Training JSON missing root keys: {sorted(missing)}")
    return data


def iter_segments(train_data: dict, include_excluded: bool = False):
    for half in train_data.get("halves", []):
        for seg in half.get("segments", []):
            if seg.get("exclusion") and not include_excluded:
                continue
            yield int(half["half"]), seg


def validate_train_manifest(train_data: dict) -> pd.DataFrame:
    rows = []
    for half in train_data.get("halves", []):
        segs = half.get("segments", [])
        for idx, seg in enumerate(segs):
            sid = seg.get("segment_id")
            recon = seg.get("reconstruction", {})
            shape = recon.get("tensor_shape", [])
            mask = recon.get("padding_mask", [])
            frames = shape[0] if len(shape) == 3 else None
            errors = []
            for key in ("start_ms", "end_ms", "duration_ms"):
                if seg.get(key, 0) % 100 != 0:
                    errors.append(f"{key}_not_100ms")
            if seg.get("end_ms") != seg.get("start_ms") + seg.get("duration_ms"):
                errors.append("end_not_start_plus_duration")
            if frames is None or shape[1:] != [23, 4]:
                errors.append("bad_tensor_shape")
            elif seg.get("duration_ms") != frames * 100:
                errors.append("duration_tensor_mismatch")
            if len(mask) != MAX_FRAMES:
                errors.append("padding_mask_len")
            elif frames is not None and int(np.sum(mask)) != frames:
                errors.append("padding_mask_sum")
            if seg.get("exclusion"):
                if seg.get("primary_team") is not None:
                    errors.append("excluded_has_primary_team")
            else:
                primary = seg.get("primary_team") or {}
                if not primary.get("intent_class"):
                    errors.append("missing_primary_intent")
                if not recon.get("npz_path"):
                    errors.append("missing_npz_path")
            if idx < len(segs) - 1 and seg.get("end_ms") != segs[idx + 1].get("start_ms"):
                errors.append("gap_or_overlap_next")
            rows.append({
                "half": half.get("half"),
                "segment_id": sid,
                "label": seg.get("exclusion") or (seg.get("primary_team") or {}).get("intent_class"),
                "start_ms": seg.get("start_ms"),
                "end_ms": seg.get("end_ms"),
                "duration_ms": seg.get("duration_ms"),
                "tensor_frames": frames,
                "npz_path": recon.get("npz_path", ""),
                "is_excluded": bool(seg.get("exclusion")),
                "errors": ";".join(errors),
            })
    return pd.DataFrame(rows)


train_data = load_train_manifest(TRAIN_JSON_PATH)
manifest_report = validate_train_manifest(train_data)
display(manifest_report)
if manifest_report["errors"].astype(bool).any():
    raise ValueError("Training manifest has schema errors. Fix before generating NPZ files.")

required_segments = list(iter_segments(train_data, include_excluded=False))
print(f"Non-excluded segments requiring NPZ: {len(required_segments)}")
print("Excluded segments skipped:", int(manifest_report["is_excluded"].sum()))

In [ ]:
# ------------------------------
# Video probe
# ------------------------------
def probe_video(path: Path) -> dict:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    return {"fps": fps, "frame_count": frame_count, "width": width, "height": height, "duration_sec": duration}

video_meta = probe_video(VIDEO_PATH)
video_meta

In [ ]:
# ------------------------------
# Load detector/tracker
# ------------------------------
if YOLO is None:
    raise ImportError("Ultralytics is not installed. Run the install cell, restart kernel, and retry.")

model = YOLO(YOLO_MODEL_NAME)
print("Loaded", YOLO_MODEL_NAME)
print("Using device", DEVICE)

# COCO class IDs used by YOLO models.
PERSON_CLASS_ID = 0
SPORTS_BALL_CLASS_ID = 32

In [ ]:
# ------------------------------
# Detection data structures
# ------------------------------
@dataclass
class Detection:
    track_id: int
    cls: int
    conf: float
    xyxy: Tuple[float, float, float, float]

    @property
    def center(self) -> Tuple[float, float]:
        x1, y1, x2, y2 = self.xyxy
        return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

    @property
    def area(self) -> float:
        x1, y1, x2, y2 = self.xyxy
        return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def parse_yolo_result(result) -> List[Detection]:
    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return []
    xyxy = boxes.xyxy.detach().cpu().numpy()
    cls = boxes.cls.detach().cpu().numpy().astype(int)
    conf = boxes.conf.detach().cpu().numpy()
    ids = boxes.id.detach().cpu().numpy().astype(int) if boxes.id is not None else np.arange(len(xyxy))
    detections = []
    for tid, c, cf, box in zip(ids, cls, conf, xyxy):
        if c == PERSON_CLASS_ID and cf >= CONF_PERSON:
            detections.append(Detection(int(tid), int(c), float(cf), tuple(map(float, box))))
        elif c == SPORTS_BALL_CLASS_ID and cf >= CONF_BALL:
            detections.append(Detection(int(tid), int(c), float(cf), tuple(map(float, box))))
    return detections


def crop_color_feature(frame: np.ndarray, det: Detection) -> np.ndarray:
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in det.xyxy]
    x1, x2 = max(0, x1), min(w, x2)
    y1, y2 = max(0, y1), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return np.array([0.0, 0.0, 0.0], dtype=np.float32)
    crop = frame[y1:y2, x1:x2]
    # Use upper torso band to reduce shorts/grass/background influence.
    ch = crop.shape[0]
    torso = crop[int(ch * 0.15): max(int(ch * 0.65), int(ch * 0.15) + 1), :]
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    mask = (sat > 35) & (val > 35)
    if int(mask.sum()) < 10:
        pixels = hsv.reshape(-1, 3)
    else:
        pixels = hsv[mask]
    return np.median(pixels, axis=0).astype(np.float32)

In [ ]:
# ------------------------------
# Team assignment and tensor packing
# ------------------------------
def assign_team_slots(frame: np.ndarray, player_dets: List[Detection], previous_slot_by_track: Dict[int, int]) -> Dict[int, int]:
    """Assign player track IDs to canonical slots 0..21.

    This is a pragmatic broadcast-video fallback. For final research runs, inspect the validation report and
    consider replacing this with a stronger jersey/team re-identification model if team colors are difficult.
    """
    if not player_dets:
        return {}

    # Preserve previous slot assignment where possible.
    assigned = {det.track_id: previous_slot_by_track[det.track_id] for det in player_dets if det.track_id in previous_slot_by_track}
    remaining = [det for det in player_dets if det.track_id not in assigned]
    used_slots = set(assigned.values())

    if not remaining:
        return assigned

    # Split remaining detections into two team groups.
    if TEAM_ASSIGNMENT_MODE == "color" and KMeans is not None and len(remaining) >= 4:
        feats = np.stack([crop_color_feature(frame, det) for det in remaining], axis=0)
        try:
            labels = KMeans(n_clusters=2, n_init=5, random_state=7).fit_predict(feats)
        except Exception:
            labels = None
    else:
        labels = None

    if labels is None:
        xs = np.array([det.center[0] for det in remaining])
        median_x = float(np.median(xs))
        labels = np.array([0 if det.center[0] <= median_x else 1 for det in remaining])

    # Stabilize team order by average x position: left group -> slots 0..10, right group -> 11..21.
    group_mean_x = []
    for group in (0, 1):
        xs = [det.center[0] for det, lab in zip(remaining, labels) if lab == group]
        group_mean_x.append(np.mean(xs) if xs else float("inf"))
    left_group = int(np.argmin(group_mean_x))

    team_groups = {0: [], 1: []}
    for det, lab in zip(remaining, labels):
        team_idx = 0 if int(lab) == left_group else 1
        team_groups[team_idx].append(det)

    for team_idx, dets in team_groups.items():
        base = 0 if team_idx == 0 else 11
        available = [slot for slot in range(base, base + 11) if slot not in used_slots]
        # Assign spatially left-to-right for determinism.
        dets = sorted(dets, key=lambda d: (d.center[0], d.center[1]))
        for det, slot in zip(dets[:len(available)], available):
            assigned[det.track_id] = slot
            used_slots.add(slot)

    return assigned


def fill_missing_positions(pos_xy: np.ndarray, visible: np.ndarray) -> np.ndarray:
    """Forward-fill then backward-fill missing xy positions per agent."""
    filled = pos_xy.copy()
    T, N, _ = filled.shape
    for agent in range(N):
        valid = visible[:, agent]
        if not valid.any():
            continue
        valid_idx = np.where(valid)[0]
        first, last = valid_idx[0], valid_idx[-1]
        for t in range(first + 1, T):
            if not valid[t, agent]:
                filled[t, agent] = filled[t - 1, agent]
        for t in range(first - 1, -1, -1):
            filled[t, agent] = filled[t + 1, agent]
    return filled


def build_trajectory_tensor(sample_times: List[float], frame_detections: List[Tuple[np.ndarray, List[Detection]]], width: int, height: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    T = len(sample_times)
    pos = np.zeros((T, MAX_AGENTS, 2), dtype=np.float32)
    conf = np.zeros((T, MAX_AGENTS), dtype=np.float32)
    visible = np.zeros((T, MAX_AGENTS), dtype=bool)
    slot_by_track: Dict[int, int] = {}

    for t, (frame, detections) in enumerate(frame_detections):
        players = sorted([d for d in detections if d.cls == PERSON_CLASS_ID], key=lambda d: d.conf, reverse=True)[:MAX_PLAYERS]
        balls = sorted([d for d in detections if d.cls == SPORTS_BALL_CLASS_ID], key=lambda d: d.conf, reverse=True)
        current_slots = assign_team_slots(frame, players, slot_by_track)
        slot_by_track.update(current_slots)

        for det in players:
            if det.track_id not in current_slots:
                continue
            slot = current_slots[det.track_id]
            cx, cy = det.center
            pos[t, slot, 0] = np.clip(cx / width, 0, 1)
            pos[t, slot, 1] = np.clip(cy / height, 0, 1)
            conf[t, slot] = det.conf
            visible[t, slot] = True

        if balls:
            ball = balls[0]
            cx, cy = ball.center
            pos[t, 22, 0] = np.clip(cx / width, 0, 1)
            pos[t, 22, 1] = np.clip(cy / height, 0, 1)
            conf[t, 22] = ball.conf
            visible[t, 22] = True

    pos_filled = fill_missing_positions(pos, visible)
    vel = np.zeros_like(pos_filled, dtype=np.float32)
    if T > 1:
        vel[1:] = (pos_filled[1:] - pos_filled[:-1]) * TARGET_FPS
    trajectory = np.concatenate([pos_filled, vel], axis=2).astype(np.float32)
    return trajectory, visible, conf

In [ ]:
# ------------------------------
# Segment processing
# ------------------------------
def sample_times_for_segment(start_ms: int, tensor_frames: int) -> List[float]:
    start_sec = start_ms / 1000.0 + VIDEO_TIME_OFFSET_SEC
    return [start_sec + i / TARGET_FPS for i in range(tensor_frames)]


def detect_segment(video_path: Path, start_ms: int, tensor_frames: int) -> Tuple[List[float], List[Tuple[np.ndarray, List[Detection]]]]:
    sample_times = sample_times_for_segment(start_ms, tensor_frames)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    frame_detections = []
    # Reset model tracker between segments to keep segment-local identities stable.
    try:
        model.predictor = None
    except Exception:
        pass

    for sample_time in sample_times:
        cap.set(cv2.CAP_PROP_POS_MSEC, sample_time * 1000.0)
        ok, frame = cap.read()
        if not ok or frame is None:
            # Keep tensor length stable; blank frame yields no detections.
            frame = np.zeros((video_meta["height"], video_meta["width"], 3), dtype=np.uint8)
            frame_detections.append((frame, []))
            continue
        result = model.track(
            frame,
            persist=True,
            imgsz=IMG_SIZE,
            conf=min(CONF_PERSON, CONF_BALL),
            iou=IOU,
            classes=[PERSON_CLASS_ID, SPORTS_BALL_CLASS_ID],
            device=DEVICE,
            verbose=False,
        )[0]
        frame_detections.append((frame, parse_yolo_result(result)))

    cap.release()
    return sample_times, frame_detections


def write_npz_for_segment(half: int, seg: dict) -> dict:
    recon = seg["reconstruction"]
    tensor_shape = recon["tensor_shape"]
    tensor_frames = int(tensor_shape[0])
    out_path = PROJECT_ROOT / recon["npz_path"]
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not OVERWRITE_EXISTING_NPZ:
        return {"segment_id": seg["segment_id"], "status": "exists", "npz_path": str(out_path)}

    sample_times, frame_detections = detect_segment(VIDEO_PATH, int(seg["start_ms"]), tensor_frames)
    trajectory, visible, conf = build_trajectory_tensor(sample_times, frame_detections, video_meta["width"], video_meta["height"])

    if list(trajectory.shape) != tensor_shape:
        raise ValueError(f"Shape mismatch for {seg['segment_id']}: got {trajectory.shape}, expected {tensor_shape}")

    player_visible_per_frame = visible[:, :22].sum(axis=1)
    ball_visible_ratio = float(visible[:, 22].mean()) if tensor_frames else 0.0
    mean_players_visible = float(player_visible_per_frame.mean()) if tensor_frames else 0.0
    min_players_visible = int(player_visible_per_frame.min()) if tensor_frames else 0
    quality_pass = bool(mean_players_visible >= 18.0 and ball_visible_ratio >= 0.20)

    np.savez_compressed(
        out_path,
        trajectory=trajectory,
        visibility_mask=visible.astype(np.uint8),
        detection_confidence=conf.astype(np.float32),
        frame_times_sec=np.array(sample_times, dtype=np.float32),
        segment_id=np.array(seg["segment_id"]),
        match_id=np.array(train_data["match_id"]),
        intent_class=np.array(seg["primary_team"]["intent_class"]),
        tensor_fps=np.array(TARGET_FPS, dtype=np.float32),
        quality_pass=np.array(quality_pass),
    )

    return {
        "segment_id": seg["segment_id"],
        "label": seg["primary_team"]["intent_class"],
        "status": "written",
        "npz_path": str(out_path),
        "frames": tensor_frames,
        "mean_players_visible": round(mean_players_visible, 2),
        "min_players_visible": min_players_visible,
        "ball_visible_ratio": round(ball_visible_ratio, 3),
        "quality_pass": quality_pass,
    }

In [ ]:
# ------------------------------
# Run NPZ generation
# ------------------------------
if DEVICE == "cpu":
    print("WARNING: running on CPU. This will be slow. Use a CUDA runtime for production generation.")

rows = []
start_time = time.time()
for half, seg in tqdm(required_segments, desc="Generating trajectory NPZ"):
    row = write_npz_for_segment(half, seg)
    rows.append(row)

report = pd.DataFrame(rows)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
report.to_csv(REPORT_PATH, index=False)
print(f"Wrote report: {REPORT_PATH}")
print(f"Elapsed: {(time.time() - start_time)/60:.1f} min")
display(report)

In [ ]:
# ------------------------------
# Final dataset validation
# ------------------------------
def validate_npz_against_manifest(train_data: dict) -> pd.DataFrame:
    rows = []
    for half, seg in iter_segments(train_data, include_excluded=False):
        npz_path = PROJECT_ROOT / seg["reconstruction"]["npz_path"]
        errors = []
        if not npz_path.exists():
            errors.append("missing_npz")
            rows.append({"segment_id": seg["segment_id"], "npz_path": str(npz_path), "errors": ";".join(errors)})
            continue
        data = np.load(npz_path, allow_pickle=False)
        if "trajectory" not in data:
            errors.append("missing_trajectory_key")
            traj = None
        else:
            traj = data["trajectory"]
            expected_shape = tuple(seg["reconstruction"]["tensor_shape"])
            if tuple(traj.shape) != expected_shape:
                errors.append(f"shape_{tuple(traj.shape)}_expected_{expected_shape}")
            if not np.isfinite(traj).all():
                errors.append("non_finite_values")
            if traj.shape[-1] == 4:
                xy = traj[:, :, :2]
                if xy.min() < -1e-6 or xy.max() > 1 + 1e-6:
                    errors.append("xy_out_of_0_1_range")
        if "visibility_mask" in data:
            vis = data["visibility_mask"]
            mean_players = float(vis[:, :22].sum(axis=1).mean())
            ball_ratio = float(vis[:, 22].mean())
        else:
            mean_players = float("nan")
            ball_ratio = float("nan")
            errors.append("missing_visibility_mask")
        rows.append({
            "segment_id": seg["segment_id"],
            "label": seg["primary_team"]["intent_class"],
            "npz_path": str(npz_path),
            "expected_shape": seg["reconstruction"]["tensor_shape"],
            "mean_players_visible": round(mean_players, 2) if np.isfinite(mean_players) else None,
            "ball_visible_ratio": round(ball_ratio, 3) if np.isfinite(ball_ratio) else None,
            "errors": ";".join(errors),
        })
    return pd.DataFrame(rows)

validation = validate_npz_against_manifest(train_data)
display(validation)

if validation["errors"].astype(bool).any():
    print("Some NPZ files failed validation. Review the rows above before training.")
else:
    print("All required NPZ files match the training manifest.")

In [ ]:
# ------------------------------
# Quick inspection helper
# ------------------------------
# Pick one segment to inspect numerically.
if len(required_segments) > 0:
    _, first_seg = required_segments[0]
    npz_path = PROJECT_ROOT / first_seg["reconstruction"]["npz_path"]
    if npz_path.exists():
        data = np.load(npz_path, allow_pickle=False)
        traj = data["trajectory"]
        print("NPZ:", npz_path)
        print("trajectory shape:", traj.shape)
        print("xy range:", float(traj[:, :, :2].min()), float(traj[:, :, :2].max()))
        print("velocity range:", float(traj[:, :, 2:].min()), float(traj[:, :, 2:].max()))
        print("visibility players mean:", float(data["visibility_mask"][:, :22].sum(axis=1).mean()))
        print("ball visible ratio:", float(data["visibility_mask"][:, 22].mean()))

## Next Step

After this notebook produces valid NPZ files, run DAG feature extraction for non-excluded segments, then train/evaluate the TACTIC-FP model in a separate notebook.

Do **not** train on segments with missing or invalid NPZ files. Missing trajectories mean the model has no real visual/player-motion input for those labels.